<a href="https://colab.research.google.com/github/hcy05020-maker/Earth-Engine/blob/GIS/Get_started_with_Earth_Engine_for_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Get started with Earth Engine for Python

In [ ]:
#@title Copyright 2024 The Earth Engine Community Authors { display-mode: "form" }
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

In [ ]:
import ee
import geemap

**2.** Authenticate and initialize the Earth Engine service. Follow the
resulting prompts to complete authentication. Be sure to replace PROJECT_ID
with the name of the project you set up for this quickstart.

In [ ]:
ee.Authenticate()
ee.Initialize(project='my-project-495906')

## Add raster data to a map

**1.** Load climate data for a given period and display its metadata.

In [ ]:
jan_2023_climate = (
    ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
    .filterDate('2023-01', '2023-02')
    .first()
)
jan_2023_climate

**2.** Instantiate a map object and add the temperature band as a layer with
specific visualization properties. Display the map.

In [ ]:
m = geemap.Map(center=[30, 0], zoom=2)

vis_params = {
    'bands': ['temperature_2m'],
    'min': 229,
    'max': 304,
    'palette': 'inferno',
}
m.add_layer(jan_2023_climate, vis_params, 'Temperature (K)')
m

## Add vector data to a map

**1.** Create a vector data object with points for three cities.

In [ ]:
cities = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point(128.28, 37.88), {'city': 'Gangwon-do'}),
    ee.Feature(ee.Geometry.Point(128.7, 36.3), {'city': 'Gyeongsangbuk-do'}),
    ee.Feature(ee.Geometry.Point(128.69, 35.23), {'city': 'Gyeongsangnam-do'}),
])
cities

**2.** Add the city locations to the map and redisplay it.

In [ ]:
m.add_layer(cities, name='Cities')
m

## Extract and chart data

**1.** Import the Altair charting library.

In [ ]:
%pip install -q --upgrade altair
import altair as alt

**2.** Extract the climate data for the three cities as a pandas DataFrame.

In [ ]:
city_climates = jan_2023_climate.reduceRegions(cities, ee.Reducer.first())

city_climates_dataframe = ee.data.computeFeatures(
    {'expression': city_climates, 'fileFormat': 'PANDAS_DATAFRAME'}
)
city_climates_dataframe

**3.** Plot the temperature for the cities as a bar chart.

In [ ]:
alt.Chart(city_climates_dataframe).mark_bar(size=100).encode(
    alt.X('city:N', sort='y', axis=alt.Axis(labelAngle=0), title='City'),
    alt.Y('temperature_2m:Q', title='Temperature (K)'),
    tooltip=[
        alt.Tooltip('city:N', title='City'),
        alt.Tooltip('temperature_2m:Q', title='Temperature (K)'),
    ],
).properties(title='January 2023 temperature for selected cities', width=500)

## What's next

  * Learn about analyzing data with Earth Engine's [objects and methods](https://developers.google.com/earth-engine/guides/objects_methods_overview).
  * Learn about Earth Engine's [processing environments](https://developers.google.com/earth-engine/guides/processing_environments).
  * Learn about Earth Engine's [machine learning capabilities](https://developers.google.com/earth-engine/guides/machine-learning).
  * Learn how to [export your computation results to BigQuery](https://developers.google.com/earth-engine/guides/exporting_to_bigquery).

## 산불 피해지 GIS 분석: 피해 강도 및 경사도

이 섹션에서는 한국의 대형 산불 피해지를 분석하여 산불 피해 강도(dNBR)와 경사도 레이어를 포함하는 GIS를 구축합니다. 2022년 울진-삼척 산불을 예시로 들어 진행합니다.

In [ ]:
# 1. 산불 지역 및 기간 정의 (2022년 울진-삼척 산불 예시)

# 울진-삼척 산불 대략적인 관심 지역(ROI) 정의
# (경도, 위도) 순서로 Point 또는 Polygon을 정의합니다.
fire_roi = ee.Geometry.Polygon([
  [129.1, 36.8],
  [129.5, 36.8],
  [129.5, 37.3], # 북쪽 경계 확장
  [129.1, 37.3], # 북쪽 경계 확장
  [129.1, 36.8]
]);

# 산불 발생 전/후 기간 정의
pre_fire_start = '2022-02-01'
pre_fire_end = '2022-02-28' # 산불 전 (2월)

post_fire_start = '2022-03-01' # 산불 발생 3월 4일경
post_fire_end = '2022-04-30' # 산불 후 (3~4월)

print(f"분석 지역: {fire_roi.getInfo()}")
print(f"산불 전 기간: {pre_fire_start} ~ {pre_fire_end}")
print(f"산불 후 기간: {post_fire_start} ~ {post_fire_end}")

In [ ]:
# 2. 위성 영상 확보 및 전처리

# Sentinel-2 Level-2A (Surface Reflectance) 영상 컬렉션 정의
# DeprecationWarning에 따라 COPERNICUS/S2_SR_HARMONIZED 사용
S2_SR_COLLECTION = 'COPERNICUS/S2_SR_HARMONIZED'

# Sentinel-2 구름 마스크 함수 정의
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).divide(10000) # SR 값을 0-1 범위로 스케일링

# NBR(Normalized Burn Ratio) 계산 함수 정의
def calculate_nbr(image):
    # NIR (B8), SWIR2 (B12) 밴드를 사용하여 NBR 계산
    # NBR = (NIR - SWIR2) / (NIR + SWIR2)
    nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR')
    return image.addBands(nbr)

In [ ]:
# 산불 전 영상 컬렉션 필터링 및 처리
pre_fire_images = (ee.ImageCollection(S2_SR_COLLECTION)
    .filterDate(pre_fire_start, pre_fire_end)
    .filterBounds(fire_roi)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) # 구름 비율 20% 미만 필터링
    .map(mask_s2_clouds) # 구름 마스크 적용
    .map(calculate_nbr)) # NBR 계산

# 가장 구름 없는 산불 전 영상 선택 (중간값)
pre_fire_image = pre_fire_images.median().clip(fire_roi)

# 산불 후 영상 컬렉션 필터링 및 처리
post_fire_images = (ee.ImageCollection(S2_SR_COLLECTION)
    .filterDate(post_fire_start, post_fire_end)
    .filterBounds(fire_roi)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) # 구름 비율 20% 미만 필터링
    .map(mask_s2_clouds) # 구름 마스크 적용
    .map(calculate_nbr)) # NBR 계산

# 가장 구름 없는 산불 후 영상 선택 (중간값)
post_fire_image = post_fire_images.median().clip(fire_roi)

print("산불 전/후 Sentinel-2 영상 및 NBR 계산 완료.")

In [ ]:
# 3. 산불 피해 강도(dNBR) 계산

# NBR 밴드가 존재하는지 확인 후 계산
if pre_fire_image.bandNames().contains('NBR') and post_fire_image.bandNames().contains('NBR'):
    # dNBR = NBR_pre - NBR_post
    dnbr = pre_fire_image.select('NBR').subtract(post_fire_image.select('NBR')).rename('dNBR')
    print("dNBR 계산 완료.")
else:
    dnbr = ee.Image(0).rename('dNBR') # NBR 밴드가 없으면 0으로 초기화 (오류 방지)
    print("경고: NBR 밴드를 찾을 수 없어 dNBR 계산을 건너뜁니다.")

# 4. 경사도(Slope) 레이어 생성

# SRTM Digital Elevation Model (DEM) 데이터 로드
dem = ee.Image('USGS/SRTMGL1_003').clip(fire_roi)

# 경사도 계산 (도 단위)
slope = ee.Terrain.slope(dem).rename('Slope_Degrees')

print("경사도 레이어 계산 완료.")

In [ ]:
# 5. GIS 레이어 시각화

# 지도 객체 생성
m = geemap.Map(center=[37.0, 129.3], zoom=9, height='600px') # 울진-삼척 지역 중심

# dNBR 시각화 파라미터 (일반적인 산불 피해 강도 분류)
# dNBR 값은 보통 -1 ~ 1 사이이며, 양수 값이 클수록 피해가 큼
dnbr_vis_params = {
    'min': -0.5,
    'max': 1.0,
    'palette': [
        '#006400',  # Green (무피해/식생증가)
        '#00FF00',  # Light Green (낮은 피해)
        '#FFFF00',  # Yellow (중간-낮은 피해)
        '#FFA500',  # Orange (중간-높은 피해)
        '#FF0000',  # Red (높은 피해)
        '#8B0000'   # Dark Red (심각한 피해)
    ]
}

# 경사도 시각화 파라미터
slope_vis_params = {
    'min': 0,
    'max': 45, # 최대 45도까지 표시
    'palette': ['lightblue', 'blue', 'darkblue'] # 경사가 높을수록 어두운 파랑
}

# 레이어를 지도에 추가
m.add_layer(pre_fire_image.select(['B4', 'B3', 'B2']), {'min': 0.0, 'max': 0.3}, 'Pre-Fire (RGB)')
m.add_layer(post_fire_image.select(['B4', 'B3', 'B2']), {'min': 0.0, 'max': 0.3}, 'Post-Fire (RGB)')
m.add_layer(dnbr, dnbr_vis_params, 'Burn Severity (dNBR)')
m.add_layer(slope, slope_vis_params, 'Slope')
m.add_colorbar(dnbr_vis_params, label="Burn Severity (dNBR)", orientation='vertical', position='bottomleft')
m.add_colorbar(slope_vis_params, label="Slope (Degrees)", orientation='vertical', position='bottomleft')

# 지도 표시
m

## 산사태 위험 지수(Landslide Risk Index, LRI) 계산 및 시각화

산사태 위험 지수는 산불 피해 지역의 경사도(Slope)와 산불 피해 강도(dNBR)를 종합하여 격자 단위로 산사태 발생 가능성을 나타냅니다. 산불로 인해 식생이 소실되면 토양 유실 및 산사태 위험이 증가하므로, dNBR 값이 높고 경사도가 가파른 지역은 더 높은 위험도를 가집니다.

여기서는 다음과 같이 간소화된 위험 수준으로 분류합니다:
- **낮은 위험 (1)**: 경사 < 15도 & dNBR < 0.2
- **중간 위험 (2)**: (15도 <= 경사 < 30도 & dNBR < 0.4) 또는 (경사 < 15도 & 0.2 <= dNBR < 0.4)
- **높은 위험 (3)**: (경사 >= 30도) 또는 (dNBR >= 0.4)
- **매우 높은 위험 (4)**: (경사 >= 40도) & (dNBR >= 0.6)

In [ ]:
# 6. 산사태 위험 지수(LRI) 계산

# LRI 계산 함수 정의
# dNBR과 Slope를 기반으로 산사태 위험도 분류 (예시)
# NBR 값은 -1 ~ 1, Slope는 0 ~ 90 (도) 범위

# dNBR 및 Slope 밴드가 있는지 확인 (없으면 기본값 사용)
dnbr_band = dnbr.select('dNBR')
slope_band = slope.select('Slope_Degrees')

# 초기 위험도 이미지 생성 (기본값 0 또는 낮은 위험)
landslide_risk = ee.Image(0).rename('Landslide_Risk_Index')

# 위험도 분류 (조건문을 사용하여 이미지 픽셀 값 할당)
# 낮은 위험 (1): 경사 < 15 AND dNBR < 0.2
landslide_risk = landslide_risk.where(
    slope_band.lt(15).And(dnbr_band.lt(0.2)),
    ee.Image(1)
)

# 중간 위험 (2): (15 <= 경사 < 30 AND dNBR < 0.4) OR (경사 < 15 AND 0.2 <= dNBR < 0.4)
landslide_risk = landslide_risk.where(
    (slope_band.gte(15).And(slope_band.lt(30)).And(dnbr_band.lt(0.4))).Or(
        slope_band.lt(15).And(dnbr_band.gte(0.2)).And(dnbr_band.lt(0.4))
    ),
    ee.Image(2)
)

# 높은 위험 (3): (경사 >= 30) OR (dNBR >= 0.4) OR (15 <= 경사 < 30 AND 0.4 <= dNBR < 1.0)
landslide_risk = landslide_risk.where(
    (slope_band.gte(30)).Or(dnbr_band.gte(0.4)).Or(
        slope_band.gte(15).And(slope_band.lt(30)).And(dnbr_band.gte(0.4)).And(dnbr_band.lt(1.0))
    ),
    ee.Image(3)
)

# 매우 높은 위험 (4): (경사 >= 40) AND (dNBR >= 0.6)
landslide_risk = landslide_risk.where(
    slope_band.gte(40).And(dnbr_band.gte(0.6)),
    ee.Image(4)
)

print("산사태 위험 지수 계산 완료.")

In [ ]:
# 7. 산사태 위험 지수 시각화

# 산사태 위험 지수 시각화 파라미터 (위험도 1~4)
landslide_risk_vis_params = {
    'min': 1,
    'max': 4,
    'palette': [
        '#00FF00',  # Low Risk (Green)
        '#FFFF00',  # Moderate Risk (Yellow)
        '#FFA500',  # High Risk (Orange)
        '#FF0000'   # Very High Risk (Red)
    ]
}

# dNBR 값을 사용하여 산불 피해 지역만 마스킹
# dNBR이 0.2 이상인 지역만 표시 (산불 피해가 있다고 간주)
burn_mask = dnbr.select('dNBR').gt(0.2) # dNBR 밴드가 0.2보다 큰 지역에만 마스크 적용
landslide_risk_masked = landslide_risk.updateMask(burn_mask)

# 레이어를 지도에 추가
m.add_layer(landslide_risk_masked, landslide_risk_vis_params, 'Landslide Risk Index (Burned Areas)')
m.add_colorbar(landslide_risk_vis_params,
               label='Landslide Risk Index (1=Low, 4=Very High)',
               orientation='vertical',
               position='bottomleft')

# 지도 표시
m

In [ ]:
import numpy as np
import ee

# Calculate approximate degree equivalent for 1km at the region's latitude
# At ~37 degrees latitude, 1 degree latitude is ~111 km, 1 degree longitude is ~89 km.
# So, 1km is roughly 0.009 degrees latitude and 0.011 degrees longitude.
lat_interval_degrees = 0.009
lon_interval_degrees = 0.011

display(m) # Ensure the map is displayed to define its bounds
# Get the current map bounds to draw the grid over the visible area
# m.get_bounds() returns a list like [west, south, east, north] in geemap 0.20.0+
bounds_list = m.get_bounds()
# If bounds_list is None or empty, use a default region or fire_roi bounds.
if bounds_list:
    west, south, east, north = bounds_list
else:
    # Fallback to fire_roi bounds if map bounds are not available
    # Note: fire_roi.getInfo() gives {'type': 'Polygon', 'coordinates': [[[129.1, 36.8], ...]]}
    # We need to extract min/max lat/lon from it.
    fire_coords = fire_roi.getInfo()['coordinates'][0]
    west = min(c[0] for c in fire_coords)
    south = min(c[1] for c in fire_coords)
    east = max(c[0] for c in fire_coords)
    north = max(c[1] for c in fire_coords)

grid_features = []

# Generate latitude lines
# Add a small buffer to ensure lines cover the visible area completely
for lat in np.arange(south - lat_interval_degrees, north + lat_interval_degrees, lat_interval_degrees):
    line = ee.Geometry.LineString([[west - lon_interval_degrees, lat], [east + lon_interval_degrees, lat]])
    grid_features.append(ee.Feature(line))

# Generate longitude lines
for lon in np.arange(west - lon_interval_degrees, east + lat_interval_degrees, lon_interval_degrees):
    line = ee.Geometry.LineString([[lon, south - lat_interval_degrees], [lon, north + lat_interval_degrees]])
    grid_features.append(ee.Feature(line))

grid_collection = ee.FeatureCollection(grid_features)

# Add the manually created grid as a layer
m.add_layer(grid_collection, {'color': 'gray'}, '1km Grid')

m

## 산림 도로 네트워크 시각화

제공해주신 Earth Engine Asset ID를 사용하여 산림 도로 네트워크를 지도에 추가합니다. 각 에셋은 `ee.FeatureCollection`으로 불러와 개별 레이어로 시각화됩니다.

In [ ]:
# 8. 산림 도로 네트워크 레이어 추가 (Asset ID 사용)

forest_road_asset_ids = [
    'projects/my-project-495906/assets/37914044',
    'projects/my-project-495906/assets/37914081',
    'projects/my-project-495906/assets/37914082',
    'projects/my-project-495906/assets/37914071',
    'projects/my-project-495906/assets/37914072'
]

# 도로 시각화 파라미터
road_vis_params = {'color': 'gray', 'width': 1}

for i, asset_id in enumerate(forest_road_asset_ids):
    try:
        # 에셋 ID로부터 FeatureCollection 로드
        road_network = ee.FeatureCollection(asset_id)
        # 지도에 레이어 추가
        m.add_layer(road_network, road_vis_params, f'Forest Road {i+1}')
        print(f'Asset {asset_id} 로드 및 지도에 추가 완료.')
    except Exception as e:
        print(f'Asset {asset_id} 로드 중 오류 발생: {e}')

# 업데이트된 지도 표시
m